# Memmingen -- MILP-Optimierung (Pyomo) *standalone Notebook*

Analog zum Stadtbach-Notebook (`20250922_Stadtbach.ipynb`): ein **einzelner
aggregierter Knoten** (kein Rohrnetz, kein Druck) mit fixen Bestandsanlagen
(CHP, Gasboiler, Biomassekessel), einer investierbaren Wärmepumpe
(einzelner Abwärmestrom `WRG_1`), einem investierbaren Elektrokessel (EK)
und einem investierbaren Wärmespeicher (TES). Löst mit Gurobi und erzeugt
KPIs, einen Dispatch-Export sowie Plots.

**Unterschiede zum Stadtbach-Notebook** (bewusst, siehe Zellenkommentare):
- Nur 1 Wärmepumpe / 1 Abwärmestrom (Memmingen hat nur eine reale WRG-Quelle,
  Stadtbach hat vier inkl. Prüfstands-/Bench-Daten).
- EK ist hier investierbar (Paper-2-Scope "WP, EK, TES"), nicht fix wie im
  Stadtbach-Notebook.
- Rohdaten sind 15-min-aufgelöst und werden auf Stundenwerte gemittelt
  (nicht summiert -- siehe Loader-Kommentar).
- Sink-Temperaturen/eta/FQ des COP-Modells kommen aus dem aktuellen,
  harmonisierten `Memmingen_P2_base.yaml`, nicht aus den älteren
  (vor-harmonisierten) Konstanten im Stadtbach-Notebook.

**Hinweise**
- Erwartet die reale Datendatei unter `../../data/Import_Data_Memmingen_epronet_cleaned.xlsx`.
- MIPGap/TimeLimit im Solve sind für interaktive Nutzung moderat gewählt;
  für publikationsreife Konvergenz siehe Kommentar in der Solve-Zelle.


In [ ]:
# 1) Imports & Versionen
# ============================
import os, sys, math, bisect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pyomo
import pyomo.environ as pyo
from pyomo.environ import (
    ConcreteModel, Var, Param, RangeSet, Set, Constraint, Objective, Expression,
    NonNegativeReals, PositiveReals, Binary, Reals, minimize, quicksum, value as pyo_val
)
from pyomo.opt import SolverFactory

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
try:
    import pyomo
    print("Pyomo:", pyomo.version.__version__)
except Exception:
    print("Pyomo: OK (Version unbekannt)")

# Zeitschrittbreite [h]: 1.0=stündlich (Rohdaten sind 15-min, werden unten gemittelt)
DT_H = 1.0

# ============================
# 2) Pfade / Konfiguration
# ============================
INPUT_XLSX = "../../data/Import_Data_Memmingen_epronet_cleaned.xlsx"
HORIZON_YEAR = 2025   # entspricht Memmingen_P2_base.yaml scenario.horizon

# Solver-Präferenzen
PREFERRED_SOLVERS = ["gurobi"]

# ============================
# 3) Loader
# ============================
def load_memmingen_xlsx(path, year=HORIZON_YEAR):
    """
    Lädt die 15-min-aufgelösten Memmingen-Rohdaten, filtert auf `year` und
    mittelt auf Stundenwerte (DT_H=1.0). Wichtig: hier wird für ALLE Spalten
    .mean() verwendet (auch für WRG1Q MW) -- das Stadtbach-Notebook summiert
    seine "Q"-Spalten beim Resampling, was für Leistungsgrößen [MW] beim
    Downsampling die Werte ~4x aufblähen würde. Für echte Energiemengen wäre
    .sum() richtig, für Leistungen [MW] ist .mean() korrekt.
    """
    df = pd.read_excel(path, usecols=[
        "Datum", "strompreis_EUR_MWh", "Waermebedarf_MWth",
        "WRG_1 °C", "WRG1Q MW",
    ])
    df["Datum"] = pd.to_datetime(df["Datum"])
    df = df[df["Datum"].dt.year == year].copy()
    df = df.set_index("Datum").sort_index()

    hourly = df.resample("h").mean()

    # Bedarf: kleine negative Ausreißer (Messrauschen) auf 0 clippen
    hourly["Waermebedarf_MWth"] = hourly["Waermebedarf_MWth"].clip(lower=0.0)
    # Preis: negative Day-Ahead-Preise sind real -> NICHT clippen
    hourly = hourly.ffill().bfill()
    return hourly

hourly = load_memmingen_xlsx(INPUT_XLSX)
T = len(hourly)
print("Stunden im Horizont:", T)

waerme       = hourly["Waermebedarf_MWth"].astype(float).tolist()
preise       = hourly["strompreis_EUR_MWh"].astype(float).tolist()
WRG1_T_list  = (hourly["WRG_1 °C"].astype(float) + 273.15).tolist()   # -> Kelvin
WRG1Q_list   = hourly["WRG1Q MW"].astype(float).clip(lower=0.0).tolist()

print("Wärmebedarf Werte:", len(waerme))
print("Strompreis Werte:", len(preise))
print("WRG1 Werte:", len(WRG1_T_list))

# ============================
# 4) COP-Logik (Lorenz-Modell, identisch zum Stadtbach-Notebook)
# Sink-Temperaturen und eta/FQ aus Memmingen_P2_base.yaml (heat_pumps.cop),
# NICHT aus dem (nicht mehr aktuellen) Stadtbach-Notebook übernommen.
# ============================
Tsink_out = 353.15   # 80°C  (Memmingen_P2_base.yaml: heat_pumps.cop.sink_defaults.Tsink_out_K)
Tsink_in  = 323.15   # 50°C  (Tsink_in_K)
deltaTpp  = 5.0       # deltaTpp_K
eta       = 0.75      # heat_pumps.types.standard.eta (harmonisiert mit Stadtbach)
FQ        = 0.10      # heat_pumps.types.standard.FQ

def lmtd(Th, Tc):
    return (Th - Tc) / np.log(Th / Tc)

LMTD_sink = lmtd(Tsink_out, Tsink_in)

Tsourcein_vals = np.linspace(273.15, 333.15, 7)   # 0..60°C (Memmingen WRG liegt niedriger als Stadtbach)
deltaT_vals    = np.array([10, 15, 20, 30])
COP_MIN_HP, COP_MAX_HP = 1.01, 12.0
_records = []
for Tsourcein in Tsourcein_vals:
    for dT in deltaT_vals:
        Tsourceout = Tsourcein - dT
        if Tsourceout <= 0 or Tsourceout >= Tsourcein:
            continue
        LMTD_source = lmtd(Tsourcein, Tsourceout)
        mdts = 0.2*(Tsink_out - Tsourceout + 2*deltaTpp) + 0.2*(Tsink_out - Tsink_in) + 0.016
        qww  = 0.0014*(Tsink_out - Tsourceout + 2*deltaTpp) - 0.0015*(Tsink_out - Tsink_in) + 0.039
        A = LMTD_sink / (LMTD_sink - LMTD_source + 1e-9)
        B = (1 + (mdts + deltaTpp)/LMTD_sink) / (1 + (mdts + 0.5*dT + 2*deltaTpp)/(LMTD_sink - LMTD_source + 1e-9))
        COP = A * B * eta * (1 - qww) + 1 - eta - FQ
        COP = float(np.clip(COP if np.isfinite(COP) else 3.0, 0.5, 12.0))
        _records.append({"Tsourcein": round(Tsourcein, 2), "Tsourceout": round(Tsourceout, 2), "COP": round(COP, 4)})

cop_lookup_df = pd.DataFrame(_records).sort_values(["Tsourcein", "Tsourceout"])

from collections import defaultdict
cop_piecewise_rules = defaultdict(list)
for _, r in cop_lookup_df.iterrows():
    cop_piecewise_rules[r["Tsourcein"]].append((r["Tsourceout"], r["COP"]))
cop_piecewise_rules = {k: sorted(v, key=lambda x: x[0]) for k, v in cop_piecewise_rules.items()}

_all_x = sorted({x for pts in cop_piecewise_rules.values() for (x, _) in pts})
x_min, x_max = (_all_x[0], _all_x[-1]) if _all_x else (0.0, 1.0)
T_grid = sorted(cop_piecewise_rules.keys())

def _interp_on_curve(pts, x):
    if x <= pts[0][0]:  return pts[0][1]
    if x >= pts[-1][0]: return pts[-1][1]
    for (x0, y0), (x1, y1) in zip(pts[:-1], pts[1:]):
        if x0 <= x <= x1:
            return y0 + (y1 - y0) * (x - x0) / (x1 - x0)
    return pts[-1][1]

def bilinear_interp_from_lookup(Tsrc_in, x, clamp_x=True):
    if not T_grid:
        return 3.0
    if clamp_x:
        x = max(x_min, min(x_max, x))
    j = bisect.bisect_left(T_grid, Tsrc_in)
    if j == 0:
        return _interp_on_curve(cop_piecewise_rules[T_grid[0]], x)
    if j == len(T_grid):
        return _interp_on_curve(cop_piecewise_rules[T_grid[-1]], x)
    t0, t1 = T_grid[j-1], T_grid[j]
    y0 = _interp_on_curve(cop_piecewise_rules[t0], x)
    y1 = _interp_on_curve(cop_piecewise_rules[t1], x)
    if not np.isfinite(y0): y0 = 3.0
    if not np.isfinite(y1): y1 = 3.0
    w = (Tsrc_in - t0) / (t1 - t0)
    val = y0 * (1 - w) + y1 * w
    return float(np.clip(val if np.isfinite(val) else 3.0, 0.5, 12.0))

def safe_Tout(Tin, dT=15.0, default_T=278.15):
    Tin = float(Tin) if np.isfinite(Tin) else default_T
    if Tin <= dT + 1e-6:
        Tin = default_T
    return max(1.0, Tin - dT)

def safe_cop(Tin, Tout, COP_MIN=COP_MIN_HP, COP_MAX=COP_MAX_HP, COP_FALLBACK=3.0):
    try:
        Tin_v, Tout_v = float(Tin), float(Tout)
    except Exception:
        return COP_FALLBACK
    x = max(x_min, min(x_max, Tout_v)) if x_min < x_max else Tout_v
    val = bilinear_interp_from_lookup(Tin_v, x) if T_grid else COP_FALLBACK
    if (not np.isfinite(val)) or (val <= 0):
        val = COP_FALLBACK
    return float(np.clip(val, COP_MIN, COP_MAX))

Tsrc_default = 15.0 + 273.15
X_default = Tsrc_default - 15.0
COPdefault_val = safe_cop(Tsrc_default, X_default)
print("COPdefault:", round(COPdefault_val, 3))
print("Beispiel COP @ WRG1[0]:", round(safe_cop(WRG1_T_list[0], safe_Tout(WRG1_T_list[0])), 3))


In [ ]:
# ============================
# 5) Model Build (lean & fast, single aggregate node)
# ============================

if len(preise) == 0 or len(waerme) == 0:
    print("[STOP] Keine harmonisierten Daten - bitte Pfade/Daten prüfen.")
    MODEL_BUILT = False
else:
    # ---------------------------
    # Anlagen-Schalter (Szenarien)
    # ---------------------------
    ENABLE_CONFIG = {
        "HP": True, "EK": True, "STORAGE": True,
        "CHP": True, "GASBOILER": True, "BIOMASS": True,
    }
    USE_RAMPS = True

    model = ConcreteModel(name="Memmingen")

    # --- Sets
    model.t = RangeSet(1, T)
    STEPS_PER_DAY = int(round(24 / DT_H))
    n_days = max(1, T // STEPS_PER_DAY)
    model.d = RangeSet(1, n_days)

    def _t_day_init(m):
        return [(d, tt)
                for d in m.d
                for tt in range((d - 1) * STEPS_PER_DAY + 1, min(d * STEPS_PER_DAY, T) + 1)]
    model.t_day = Set(dimen=2, initialize=_t_day_init)

    # --- Ökonomie & Preise (Memmingen_P2_base.yaml: grid / fuels / costs)
    model.strompreis   = Param(model.t, initialize={i: preise[i-1] for i in range(1, T+1)}, default=0.0)
    model.waermebedarf = Param(model.t, initialize={i: waerme[i-1]  for i in range(1, T+1)}, default=0.0)

    model.Leistungspreis      = Param(initialize=127240.0)   # demand_charge_eur_per_mw_y
    model.Gritcost            = Param(initialize=25.0)       # gridcost_eur_mwh
    model.Gaspreis            = Param(initialize=45.0)       # fuels.gas.price_eur_mwh
    model.Biomassepreis       = Param(initialize=40.0)       # fuels.biomass.price_eur_mwh

    model.einspeisepreis = Param(initialize=0.0, within=NonNegativeReals, mutable=True)
    model.SELL_HAIRCUT = Param(initialize=0.05, mutable=True)
    model.SELL_SPREAD  = Param(initialize=5.0,  mutable=True)
    model.SELL_FEE     = Param(initialize=5.0,  mutable=True)
    model.SELL_PREMIUM = Param(initialize=0.0,  mutable=True)

    model.sell_base = Expression(
        model.t,
        rule=lambda m, t: (1.0 - m.SELL_HAIRCUT) * m.strompreis[t]
                          + m.SELL_PREMIUM - m.SELL_FEE - m.SELL_SPREAD
    )
    model.sell_price_eff = pyo.Param(
        model.t,
        initialize=lambda m, t: max(float(pyo_val(m.sell_base[t])), float(pyo_val(m.einspeisepreis))),
        mutable=True
    )

    def refresh_sell_price_eff(m):
        floor = float(pyo_val(m.einspeisepreis))
        for t in m.t:
            base_t = float(pyo_val(m.sell_base[t]))
            m.sell_price_eff[t].set_value(max(base_t, floor))
    model.refresh_sell_price_eff = refresh_sell_price_eff

    # --------------------------
    # Enable-Parameter (mutable)
    # --------------------------
    model.EN_HP        = Param(initialize=int(ENABLE_CONFIG["HP"]),        within=NonNegativeReals, mutable=True)
    model.EN_EK        = Param(initialize=int(ENABLE_CONFIG["EK"]),        within=NonNegativeReals, mutable=True)
    model.EN_STORAGE   = Param(initialize=int(ENABLE_CONFIG["STORAGE"]),   within=NonNegativeReals, mutable=True)
    model.EN_CHP       = Param(initialize=int(ENABLE_CONFIG["CHP"]),       within=NonNegativeReals, mutable=True)
    model.EN_GASBOILER = Param(initialize=int(ENABLE_CONFIG["GASBOILER"]), within=NonNegativeReals, mutable=True)
    model.EN_BIOMASS   = Param(initialize=int(ENABLE_CONFIG["BIOMASS"]),   within=NonNegativeReals, mutable=True)

    # ---------------------------------------------------------------------
    # TES (tes_main): continuous capacity Var (not the framework's discrete
    # ladder) sized from real physical/economic parameters in
    # Memmingen_P2_base.yaml's tes_main block, not an arbitrary placeholder.
    # E[MWh] = V[m3] * rho*cp[kWh/m3K] * dT[K] / 1000
    # ---------------------------------------------------------------------
    WATER_KWH_PER_M3_K = 1.1628          # rho=1000 kg/m3, cp=4.186 kJ/kgK -> kWh/(m3*K)
    TES_V_MAX_M3        = 5000.0         # tes_main.V_max_m3
    TES_DELTA_T_K        = 15.0          # network.min_supply_delta_T_k (worst-case corridor)
    TES_ALPHA_EUR_M3     = 1200.0        # tes_main.alpha_tes_eur_per_m3
    TES_BETA_EUR         = 100000.0      # tes_main.beta_tes_eur (fixed CAPEX per tank)
    TES_LIFETIME_YEARS   = 30.0

    STO_ENERGY_DENSITY_MWH_PER_M3 = WATER_KWH_PER_M3_K * TES_DELTA_T_K / 1000.0
    STO_E_MAX = TES_V_MAX_M3 * STO_ENERGY_DENSITY_MWH_PER_M3               # ~87.2 MWh
    CAPEX_TES_EUR_PER_MWH = TES_ALPHA_EUR_M3 / STO_ENERGY_DENSITY_MWH_PER_M3  # ~68.8 k€/MWh
    STO_E_MIN = 1.0   # MWh, minimum tank size if built
    STO_POWER_TO_ENERGY = 0.25           # tes_main.power_to_energy_ratio

    model.storage_eff_charge    = Param(initialize=0.98)   # tes_main.eff_charge
    model.storage_eff_discharge = Param(initialize=0.98)   # tes_main.eff_discharge
    model.storage_loss_hour     = Param(initialize=0.9999, mutable=True)
    model.storage_loss_step     = pyo.Expression(rule=lambda m: m.storage_loss_hour ** DT_H)
    model.CAPEXspeicher         = Param(initialize=CAPEX_TES_EUR_PER_MWH, within=PositiveReals, mutable=True)
    model.TESInstallationskosten = Param(initialize=TES_BETA_EUR)
    model.Lebensdauerspeicher   = Param(initialize=TES_LIFETIME_YEARS)

    model.storage_capacity        = Var(bounds=(0, STO_E_MAX))
    model.storage_power           = Var(bounds=(0, STO_POWER_TO_ENERGY * STO_E_MAX))
    model.storage_capacity_active = Var(domain=Binary)
    model.storage_level      = Var(model.t, domain=NonNegativeReals)
    model.storage_charge     = Var(model.t, domain=NonNegativeReals)
    model.storage_discharge  = Var(model.t, domain=NonNegativeReals)
    model.SOC_init           = Param(initialize=0.0, within=NonNegativeReals, mutable=True)
    model.sto_mode           = Var(model.t, domain=Binary)   # 1=Laden, 0=Entladen

    model.sto_mode_gate = pyo.Constraint(model.t, rule=lambda m, t: m.sto_mode[t] <= m.storage_capacity_active)
    model.storage_power_rate = Constraint(expr=model.storage_power <= STO_POWER_TO_ENERGY * model.storage_capacity)
    model.sto_gate_charge    = Constraint(model.t, rule=lambda m, t: m.storage_charge[t]    <= m.storage_power * m.EN_STORAGE * m.sto_mode[t])
    model.sto_gate_discharge = Constraint(model.t, rule=lambda m, t: m.storage_discharge[t] <= m.storage_power * m.EN_STORAGE * (1 - m.sto_mode[t]))
    model.soc_cap             = Constraint(model.t, rule=lambda m, t: m.storage_level[t]     <= m.storage_capacity * m.EN_STORAGE)
    model.storage_linkE_hi = Constraint(expr=model.storage_capacity <= STO_E_MAX * model.storage_capacity_active * model.EN_STORAGE)
    model.storage_linkE_lo = Constraint(expr=model.storage_capacity >= STO_E_MIN * model.storage_capacity_active * model.EN_STORAGE)
    model.storage_active_gate = Constraint(expr=model.storage_capacity_active <= model.EN_STORAGE)

    def storage_state_rule(m, t):
        prev = (m.storage_level[t-1] * m.storage_loss_step) if t > m.t.first() else m.SOC_init
        return prev + m.storage_charge[t] * DT_H - m.storage_discharge[t] * DT_H == m.storage_level[t]
    model.storage_state = Constraint(model.t, rule=storage_state_rule)
    model.soc_terminal   = pyo.Constraint(expr=model.storage_level[model.t.last()] >= model.SOC_init)

    # ---------------------------------------------------------------------
    # Heat pump (hp_main) — single unit, single waste-heat stream (WRG1)
    # ---------------------------------------------------------------------
    model.LebensdauerHP          = Param(initialize=20.0)   # hp_main.investment.lifetime_years
    model.CapexHP                = Param(initialize=700000.0, within=PositiveReals, mutable=True)  # capex_eur_per_mw
    model.HPInstallationskosten  = Param(initialize=50000.0)  # activation_cost_eur
    model.HPstarts               = Param(initialize=10)
    model.HP_min_load_fraction   = Param(initialize=0.20)   # hp_main.min_load
    model.HP_ramp_fraction_per_h = Param(initialize=0.50)

    HP_MAX = 30.0   # hp_main.investment.capacity_max_mw
    HP_MIN = 1.0

    model.WRG1T = Param(model.t, initialize={i: WRG1_T_list[i-1] for i in range(1, T+1)}, within=Reals, default=278.15, mutable=True)
    model.WRG1Q = Param(model.t, initialize={i: WRG1Q_list[i-1]  for i in range(1, T+1)}, within=NonNegativeReals, default=0.0, mutable=True)

    model.Tsourceout1 = Param(model.t, initialize={i: safe_Tout(WRG1_T_list[i-1]) for i in range(1, T+1)}, within=PositiveReals)
    model.COP1        = Param(model.t, initialize={i: safe_cop(WRG1_T_list[i-1], safe_Tout(WRG1_T_list[i-1])) for i in range(1, T+1)}, within=PositiveReals)
    model.COPdefault  = Param(initialize=COPdefault_val, within=PositiveReals)

    model.HPNenn        = Var(bounds=(0, HP_MAX))
    model.HPNenn_active  = Var(domain=Binary)
    model.onHP           = Var(model.t, domain=Binary)

    model.Qwrg1 = Var(model.t, domain=NonNegativeReals)
    model.Qdef1 = Var(model.t, domain=NonNegativeReals)
    model.Qhp1  = Var(model.t, domain=NonNegativeReals)

    model.wrg1_cap = Constraint(model.t, rule=lambda m, t: m.Qwrg1[t] <= m.WRG1Q[t])
    model.hp1_balance = Constraint(model.t, rule=lambda m, t: m.Qhp1[t] == m.Qwrg1[t] + m.Qdef1[t])

    model.hp1_cap1 = Constraint(model.t, rule=lambda m, t: m.Qhp1[t] <= m.HPNenn)
    model.hp1_cap2 = Constraint(model.t, rule=lambda m, t: m.Qhp1[t] <= HP_MAX * m.onHP[t])
    model.hp1_min  = Constraint(model.t, rule=lambda m, t:
        m.Qhp1[t] >= m.HP_min_load_fraction * m.HPNenn - (1 - m.onHP[t]) * m.HP_min_load_fraction * HP_MAX)

    model.on_impl_hp = Constraint(model.t, rule=lambda m, t: m.onHP[t] <= m.HPNenn_active)
    model.hp_link_hi = Constraint(expr=model.HPNenn <= HP_MAX * model.HPNenn_active * model.EN_HP)
    model.hp_link_lo = Constraint(expr=model.HPNenn >= HP_MIN * model.HPNenn_active * model.EN_HP)
    model.hp_active_gate = Constraint(expr=model.HPNenn_active <= model.EN_HP)
    model.hp_enable_on   = Constraint(model.t, rule=lambda m, t: m.onHP[t] <= m.EN_HP)

    if USE_RAMPS:
        model.HP_RampUp = pyo.Constraint(model.t, rule=lambda m, t:
            pyo.Constraint.Skip if t == m.t.first() else m.Qhp1[t] - m.Qhp1[t-1] <= m.HP_ramp_fraction_per_h * m.HPNenn * DT_H)
        model.HP_RampDown = pyo.Constraint(model.t, rule=lambda m, t:
            pyo.Constraint.Skip if t == m.t.first() else m.Qhp1[t-1] - m.Qhp1[t] <= m.HP_ramp_fraction_per_h * m.HPNenn * DT_H)

    model.start_hp = Var(model.t, domain=Binary)
    model.start_hp_a = Constraint(model.t, rule=lambda m, t: (m.start_hp[t] == m.onHP[t]) if t == m.t.first() else pyo.Constraint.Skip)
    model.start_hp_b = Constraint(model.t, rule=lambda m, t: (m.start_hp[t] >= m.onHP[t] - m.onHP[t-1]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start_hp_c = Constraint(model.t, rule=lambda m, t: (m.start_hp[t] <= m.onHP[t]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start_hp_d = Constraint(model.t, rule=lambda m, t: (m.start_hp[t] <= 1 - m.onHP[t-1]) if t > m.t.first() else pyo.Constraint.Skip)
    model.daily_starts_hp = Constraint(model.d, rule=lambda m, d: pyo.quicksum(m.start_hp[t] for (dd, t) in m.t_day if dd == d) <= m.HPstarts)

    # ---------------------------------------------------------------------
    # Electric boiler (eboiler_main / EK) — now investable per Paper 2 scope.
    # Simpler than the HP: capacity Var + activation binary, no start/ramp
    # constraints (not physically meaningful for a resistive boiler).
    # ---------------------------------------------------------------------
    model.LebensdauerEK         = Param(initialize=25.0)   # eboiler_main.investment.lifetime_years
    model.CapexEK               = Param(initialize=150000.0, within=PositiveReals, mutable=True)  # capex_eur_per_mw
    model.EKInstallationskosten = Param(initialize=20000.0)  # activation_cost_eur
    model.EK_eff                = Param(initialize=0.99)     # eboiler_main.efficiency

    EK_MAX = 20.0   # eboiler_main.investment.capacity_max_mw
    EK_MIN = 0.5

    model.EKNenn        = Var(bounds=(0, EK_MAX))
    model.EKNenn_active  = Var(domain=Binary)
    model.EK_Power       = Var(model.t, domain=NonNegativeReals)

    model.ek_cap    = Constraint(model.t, rule=lambda m, t: m.EK_Power[t] <= m.EKNenn * m.EN_EK)
    model.ek_link_hi = Constraint(expr=model.EKNenn <= EK_MAX * model.EKNenn_active * model.EN_EK)
    model.ek_link_lo = Constraint(expr=model.EKNenn >= EK_MIN * model.EKNenn_active * model.EN_EK)
    model.ek_active_gate = Constraint(expr=model.EKNenn_active <= model.EN_EK)

    # ---------------------------------------------------------------------
    # Fixed conventional units (existing infrastructure, NOT investable)
    # ---------------------------------------------------------------------
    model.CHP_th_eff       = Param(initialize=0.80)
    model.CHP_el_eff       = Param(initialize=0.40)
    model.GASBOILER_th_eff = Param(initialize=0.90)
    model.BIOMASS_th_eff   = Param(initialize=0.85)

    model.CHP_Power       = Var(model.t, domain=NonNegativeReals, bounds=(0, 0.20))
    model.GASBOILER_Power = Var(model.t, domain=NonNegativeReals, bounds=(0, 13.0))
    model.BIOMASS_Power   = Var(model.t, domain=NonNegativeReals, bounds=(0, 3.3))

    model.chp_gate       = Constraint(model.t, rule=lambda m, t: m.CHP_Power[t]       <= 0.20 * m.EN_CHP)
    model.gasboiler_gate = Constraint(model.t, rule=lambda m, t: m.GASBOILER_Power[t] <= 13.0 * m.EN_GASBOILER)
    model.biomass_gate   = Constraint(model.t, rule=lambda m, t: m.BIOMASS_Power[t]   <= 3.3  * m.EN_BIOMASS)

    # --- Wärme-Erzeugung & Bilanz
    model.total_heat = pyo.Expression(
        model.t,
        rule=lambda m, t: (
            m.Qhp1[t]
          + m.EK_Power[t]         * m.EK_eff            * m.EN_EK
          + m.CHP_Power[t]        * m.CHP_th_eff         * m.EN_CHP
          + m.GASBOILER_Power[t]  * m.GASBOILER_th_eff   * m.EN_GASBOILER
          + m.BIOMASS_Power[t]    * m.BIOMASS_th_eff     * m.EN_BIOMASS
        )
    )
    model.waerme_demand = pyo.Constraint(
        model.t,
        rule=lambda m, t:
            m.total_heat[t]
          - m.storage_charge[t]    / m.storage_eff_charge
          + m.storage_discharge[t] * m.storage_eff_discharge
          >= m.waermebedarf[t]
    )

    # --- Elektrizitätsbilanz (nur CHP erzeugt Strom; Memmingen hat kein GTOST/BMHKW-CHP)
    P_FLOW_CAP = 100.0
    model.P_buy  = Var(model.t, domain=NonNegativeReals, bounds=(0, P_FLOW_CAP))
    model.P_sell = Var(model.t, domain=NonNegativeReals, bounds=(0, P_FLOW_CAP))

    model.e_balance = Constraint(
        model.t,
        rule=lambda m, t:
            (m.CHP_Power[t] * m.CHP_el_eff * m.EN_CHP + m.P_buy[t])
          - (m.Qwrg1[t]/m.COP1[t] + m.Qdef1[t]/m.COPdefault + m.EK_Power[t] * m.EN_EK + m.P_sell[t])
          == 0
    )

    model.Gasverbrauch = pyo.Expression(
        model.t,
        rule=lambda m, t: m.CHP_Power[t] * m.EN_CHP + m.GASBOILER_Power[t] * m.EN_GASBOILER
    )

    model.max_stromverbrauch = Var(domain=NonNegativeReals)
    model.max_stromverbrauch_constraint = Constraint(model.t, rule=lambda m, t: m.max_stromverbrauch >= m.P_buy[t])

    model.buy_price = pyo.Expression(model.t, rule=lambda m, t: m.strompreis[t] + m.Gritcost)

    model.M_GRID    = Param(initialize=P_FLOW_CAP)
    model.grid_mode = Var(model.t, domain=Binary)
    model.buy_gate  = Constraint(model.t, rule=lambda m, t: m.P_buy[t]  <= m.M_GRID * m.grid_mode[t])
    model.sell_gate = Constraint(model.t, rule=lambda m, t: m.P_sell[t] <= m.M_GRID * (1 - m.grid_mode[t]))

    # --- Objective (straight-line CAPEX annualization, same style as the Stadtbach notebook)
    HOURS_PER_YEAR = 8760.0
    model.year_frac = Param(initialize=float(T) * DT_H / HOURS_PER_YEAR)

    model.total_cost = Objective(
        expr=(
              pyo.quicksum(DT_H * model.P_buy[t]  * model.buy_price[t]      for t in model.t)
            - pyo.quicksum(DT_H * model.P_sell[t] * model.sell_price_eff[t] for t in model.t)
            + pyo.quicksum(DT_H * model.Gasverbrauch[t]   * model.Gaspreis      for t in model.t)
            + pyo.quicksum(DT_H * model.BIOMASS_Power[t]  * model.Biomassepreis for t in model.t)
            + model.HPNenn * model.CapexHP / model.LebensdauerHP * model.year_frac
            + model.HPNenn_active * model.HPInstallationskosten / model.LebensdauerHP * model.year_frac
            + model.EKNenn * model.CapexEK / model.LebensdauerEK * model.year_frac
            + model.EKNenn_active * model.EKInstallationskosten / model.LebensdauerEK * model.year_frac
            + model.CAPEXspeicher * model.storage_capacity / model.Lebensdauerspeicher * model.year_frac
            + model.storage_capacity_active * model.TESInstallationskosten / model.Lebensdauerspeicher * model.year_frac
            + model.Leistungspreis * model.max_stromverbrauch * model.year_frac
        ),
        sense=pyo.minimize
    )

    MODEL_BUILT = True
    print("Model built with T =", T)
    print("STO_E_MAX [MWh]:", round(STO_E_MAX, 2), " CAPEX_TES [EUR/MWh]:", round(CAPEX_TES_EUR_PER_MWH, 0))


In [ ]:
# ============================
# 6) Solve
# ============================
# MIPGap/TimeLimit here are relaxed for interactive notebook use. The full
# framework's Paper-2 campaign config (Memmingen_P2_base.yaml) uses
# MIPGap=0.005 / TimeLimit=86400 (24h) for publication-grade convergence --
# tighten these the same way if you need directly comparable results.
solver = SolverFactory('gurobi')
solver.options.update({
    'MIPGap': 0.01,
    'TimeLimit': 3600,
    'Cuts': 2,
    'MIPFocus': 2,
    'Heuristics': 0.1,
    'NumericFocus': 2,
    'OutputFlag': 1,
    'LogToConsole': 1,
})

result = solver.solve(model, tee=True, load_solutions=True)

ok_tc = {
    pyo.TerminationCondition.optimal,
    pyo.TerminationCondition.locallyOptimal,
    pyo.TerminationCondition.feasible,
    pyo.TerminationCondition.maxTimeLimit,
}
# NOTE: on a TimeLimit abort, Gurobi/Pyomo report solver.status == 'aborted'
# even though a usable incumbent is loaded (load_solutions=True). Checking
# termination_condition alone (not solver.status) is what correctly detects
# that a "hit the clock" run still has a usable near-optimal solution --
# the Stadtbach notebook's status-only check misses this case.
solved_ok = result.solver.termination_condition in ok_tc
print("Solver status:", result.solver.status, "| Termination:", result.solver.termination_condition)

# ============================
# 7) KPIs
# ============================
val = pyo_val

if solved_ok:
    dt = DT_H
    HPNenn_MW = val(model.HPNenn)
    EKNenn_MW = val(model.EKNenn)
    TES_MWh   = val(model.storage_capacity)

    E_hp   = sum(val(model.Qhp1[t]) for t in model.t) * dt
    E_ek   = sum(val(model.EK_Power[t]) * val(model.EK_eff) for t in model.t) * dt
    E_chp  = sum(val(model.CHP_Power[t]) * val(model.CHP_th_eff) for t in model.t) * dt
    E_gb   = sum(val(model.GASBOILER_Power[t]) * val(model.GASBOILER_th_eff) for t in model.t) * dt
    E_bio  = sum(val(model.BIOMASS_Power[t]) * val(model.BIOMASS_th_eff) for t in model.t) * dt
    E_total_heat = E_hp + E_ek + E_chp + E_gb + E_bio
    E_demand = sum(val(model.waermebedarf[t]) for t in model.t) * dt

    total_buy_cost = sum(dt * val(model.P_buy[t])  * val(model.buy_price[t])      for t in model.t)
    total_sell_rev = sum(dt * val(model.P_sell[t]) * val(model.sell_price_eff[t]) for t in model.t)
    total_gas_cost = sum(dt * val(model.Gasverbrauch[t]) * val(model.Gaspreis)    for t in model.t)
    total_bio_cost = sum(dt * val(model.BIOMASS_Power[t]) * val(model.Biomassepreis) for t in model.t)

    hp_capex  = HPNenn_MW * val(model.CapexHP) / val(model.LebensdauerHP) * val(model.year_frac)
    hp_inst   = val(model.HPNenn_active) * val(model.HPInstallationskosten) / val(model.LebensdauerHP) * val(model.year_frac)
    ek_capex  = EKNenn_MW * val(model.CapexEK) / val(model.LebensdauerEK) * val(model.year_frac)
    ek_inst   = val(model.EKNenn_active) * val(model.EKInstallationskosten) / val(model.LebensdauerEK) * val(model.year_frac)
    tes_capex = TES_MWh * val(model.CAPEXspeicher) / val(model.Lebensdauerspeicher) * val(model.year_frac)
    tes_inst  = val(model.storage_capacity_active) * val(model.TESInstallationskosten) / val(model.Lebensdauerspeicher) * val(model.year_frac)
    demand_charge_cost = val(model.Leistungspreis) * val(model.max_stromverbrauch) * val(model.year_frac)

    print("\n=== Investable capacities ===")
    print(f"  HP  (hp_main):      {HPNenn_MW:8.2f} MW_th")
    print(f"  EK  (eboiler_main): {EKNenn_MW:8.2f} MW_th")
    print(f"  TES (tes_main):     {TES_MWh:8.2f} MWh  (max bound {STO_E_MAX:.1f} MWh)")

    print("\n=== Heat balance ===")
    print(f"  Demand:   {E_demand:10.1f} MWh_th")
    print(f"  Supplied: {E_total_heat:10.1f} MWh_th  (HP {E_hp:.0f} / EK {E_ek:.0f} / CHP {E_chp:.0f} / Gasboiler {E_gb:.0f} / Biomass {E_bio:.0f})")

    print("\n=== Cost breakdown [EUR/a] ===")
    print(f"  Grid buy:       {total_buy_cost:12,.0f}")
    print(f"  Grid sell:      {-total_sell_rev:12,.0f}")
    print(f"  Gas fuel:       {total_gas_cost:12,.0f}")
    print(f"  Biomass fuel:   {total_bio_cost:12,.0f}")
    print(f"  HP CAPEX+inst:  {hp_capex + hp_inst:12,.0f}")
    print(f"  EK CAPEX+inst:  {ek_capex + ek_inst:12,.0f}")
    print(f"  TES CAPEX+inst: {tes_capex + tes_inst:12,.0f}")
    print(f"  Demand charge:  {demand_charge_cost:12,.0f}")
    print(f"  TOTAL (model):  {val(model.total_cost):12,.0f}")

    heat_price_eur_kwh = val(model.total_cost) / (E_total_heat * 1000.0) if E_total_heat > 0 else float("nan")
    print(f"\n  Implied heat price: {heat_price_eur_kwh*100:.2f} ct/kWh_th")

    # ============================
    # 8) Timeseries-Export
    # ============================
    idx = pd.date_range("2025-01-01", periods=T, freq="h")

    dispatch = pd.DataFrame({
        "Q_HP_MWth":    [val(model.Qhp1[t]) for t in model.t],
        "Qwrg1_MWth":   [val(model.Qwrg1[t]) for t in model.t],
        "Qdef1_MWth":   [val(model.Qdef1[t]) for t in model.t],
        "CHP_Fuel_MW":  [val(model.CHP_Power[t]) for t in model.t],
        "Gasboiler_Fuel_MW": [val(model.GASBOILER_Power[t]) for t in model.t],
        "Biomass_Fuel_MW":   [val(model.BIOMASS_Power[t]) for t in model.t],
        "EK_El_MWel":   [val(model.EK_Power[t]) for t in model.t],
        "Sto_Charge_MWth":    [val(model.storage_charge[t]) for t in model.t],
        "Sto_Discharge_MWth": [val(model.storage_discharge[t]) for t in model.t],
        "Sto_SoC_MWhth":      [val(model.storage_level[t]) for t in model.t],
        "P_buy_MWel":  [val(model.P_buy[t]) for t in model.t],
        "P_sell_MWel": [val(model.P_sell[t]) for t in model.t],
        "Waermebedarf_MWth": [val(model.waermebedarf[t]) for t in model.t],
    }, index=idx)
    dispatch["CHP_Heat_MWth"]       = dispatch["CHP_Fuel_MW"] * val(model.CHP_th_eff)
    dispatch["CHP_Power_MWel"]      = dispatch["CHP_Fuel_MW"] * val(model.CHP_el_eff)
    dispatch["Gasboiler_Heat_MWth"] = dispatch["Gasboiler_Fuel_MW"] * val(model.GASBOILER_th_eff)
    dispatch["Biomass_Heat_MWth"]   = dispatch["Biomass_Fuel_MW"] * val(model.BIOMASS_th_eff)
    dispatch["EK_Heat_MWth"]        = dispatch["EK_El_MWel"] * val(model.EK_eff)

    dispatch.to_excel("Memmingen_dispatch_timeseries.xlsx", sheet_name="Dispatch_MW")
    print("\nExport: Memmingen_dispatch_timeseries.xlsx geschrieben.")

    # ============================
    # 9) Plots
    # ============================
    week = dispatch.loc["2025-01-15":"2025-01-22"]
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.stackplot(
        week.index,
        week["CHP_Heat_MWth"], week["Gasboiler_Heat_MWth"], week["Biomass_Heat_MWth"],
        week["Q_HP_MWth"], week["EK_Heat_MWth"], week["Sto_Discharge_MWth"],
        labels=["CHP", "Gasboiler", "Biomass", "Heat pump", "EK", "TES discharge"],
    )
    ax.plot(week.index, week["Waermebedarf_MWth"] + week["Sto_Charge_MWth"], "k--", label="Demand (+ TES charge)")
    ax.set_ylabel("MW_th")
    ax.set_title("Memmingen dispatch — representative week (2025-01-15 .. 01-22)")
    ax.legend(loc="upper left", ncol=3, fontsize=8)
    fig.tight_layout()
    fig.savefig("Memmingen_dispatch_week.png", dpi=150)
    plt.show()

    cost_breakdown = pd.Series({
        "Grid buy": total_buy_cost,
        "Grid sell": -total_sell_rev,
        "Gas fuel": total_gas_cost,
        "Biomass fuel": total_bio_cost,
        "HP CAPEX+inst": hp_capex + hp_inst,
        "EK CAPEX+inst": ek_capex + ek_inst,
        "TES CAPEX+inst": tes_capex + tes_inst,
        "Demand charge": demand_charge_cost,
    })
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    cost_breakdown.plot.bar(ax=ax2, color=["#4C72B0" if v >= 0 else "#55A868" for v in cost_breakdown])
    ax2.set_ylabel("EUR / a")
    ax2.set_title("Memmingen annual cost breakdown")
    fig2.tight_layout()
    fig2.savefig("Memmingen_cost_breakdown.png", dpi=150)
    plt.show()
else:
    print("[WARN] Keine verwertbare Lösung.")
